# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## ML Process

Based on the knowledge gathered from the experiments, this notebook aims to create 3 models.

## Preparing the data

### Transforming the csv data to a numpy array

In [1]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

usdYen_raw_data = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
usdYen_raw_data_dates = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})

print("Length: ",len(usdYen_raw_data))
print("Data type: ",usdYen_raw_data.dtype)
print("Raw Data: ",usdYen_raw_data)
print("Raw Data Dates: ",usdYen_raw_data_dates)

Length:  5000
Data type:  float64
Raw Data:  [154.71 155.21 155.81 ... 118.22 118.89 118.46]
Raw Data Dates:  ['2025-12-16' '2025-12-15' '2025-12-12' ... '2006-10-19' '2006-10-18'
 '2006-10-17']


As the currency data is from newer to older, the order should be inverted.

In [2]:
usdYen_raw_data = np.flip(usdYen_raw_data, axis=0)
print(usdYen_raw_data)

[118.46 118.89 118.22 ... 155.81 155.21 154.71]


### Computing the numer of samples for each data split

In [3]:
train_samples_number = len(usdYen_raw_data)
print("Number of train samples: ", train_samples_number)

Number of train samples:  5000


### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [4]:
from tensorflow import keras

# Parameters
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delay = sampling_rate * (sequence_length + 5 - 1) # target is 5 days after the end of the sequence
batch_size = train_samples_number

# train dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    usdYen_raw_data,
    targets=usdYen_raw_data[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

### - Checking that timeseries data works correctly

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [5]:
'''
for inputs, targets in train_dataset:
    for i in range(inputs.shape[0]):
        print([float(x) for x in inputs[i]], float(targets[i]))
'''

'\nfor inputs, targets in train_dataset:\n    for i in range(inputs.shape[0]):\n        print([float(x) for x in inputs[i]], float(targets[i]))\n'

### - Extracting data inputs and outputs

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [6]:
import tensorflow as tf

data_inputs = []
data_outputs = []

for samples, targets in train_dataset:
    print("Samples: ", samples)
    print("Sample shape: ", samples.shape)
    print("Targets: ", targets)
    print("Targets shape: ", targets.shape)
    data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
    data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

data_inputs_test = data_inputs[-200:]
data_outputs_test = data_outputs[-200:]
data_inputs = data_inputs[:-200]
data_outputs = data_outputs[:-200]

print("Data Inputs: ", len(data_inputs))
print("Data Inputs Test: ", len(data_inputs_test))
print("Data Outputs: ", len(data_outputs))
print("Data Outputs Test: ", len(data_outputs_test))
    

Samples:  tf.Tensor(
[[118.46  118.89  118.22  ... 119.88  120.18  120.34 ]
 [118.89  118.22  118.7   ... 120.18  120.34  120.26 ]
 [118.22  118.7   119.34  ... 120.34  120.26  120.8  ]
 ...
 [148.35  147.453 146.63  ... 155.24  155.08  155.34 ]
 [147.453 146.63  145.592 ... 155.08  155.34  155.92 ]
 [146.63  145.592 145.581 ... 155.34  155.92  156.86 ]], shape=(4846, 150), dtype=float64)
Sample shape:  (4846, 150)
Targets:  tf.Tensor([121.45 121.54 121.65 ... 155.81 155.21 154.71], shape=(4846,), dtype=float64)
Targets shape:  (4846,)
Data Inputs:  4646
Data Inputs Test:  200
Data Outputs:  4646
Data Outputs Test:  200


2026-01-21 16:30:56.440502: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [7]:
print("----")
print("Input Data: ", data_inputs)
print("----")
print("Output Data: ", data_outputs)
print("----")

----
Input Data:  [[118.46  118.89  118.22  ... 119.88  120.18  120.34 ]
 [118.89  118.22  118.7   ... 120.18  120.34  120.26 ]
 [118.22  118.7   119.34  ... 120.34  120.26  120.8  ]
 ...
 [144.052 144.518 146.702 ... 149.076 149.801 150.545]
 [144.518 146.702 147.181 ... 149.801 150.545 149.485]
 [146.702 147.181 146.634 ... 150.545 149.485 149.801]]
----
Output Data:  tf.Tensor(
[[121.45 ]
 [121.54 ]
 [121.65 ]
 ...
 [148.018]
 [147.151]
 [147.708]], shape=(4646, 1), dtype=float64)
----


## Baseline: Basic machine learning model

The baseline will be based on the code provided in Chapter 10.2.3 (Deep Learning For Timeseries, Let’s try a basic machine learning model) of the book Deep Learning with Python by Francois Chollet [2].

In [8]:
from keras import models
from keras import layers
from keras import activations

def build_baseline_basic_model():
    model = models.Sequential()
    model.add(layers.Dense(16, input_shape=(sequence_length, 1),activation=activations.relu))
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

baseline_basic_model = build_baseline_basic_model()

/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 150, 16)        │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 150, 1)         │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49 (196.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

Keras checkpoint to save model with best performance

In [9]:
baseline_basic_model_callbacks = [
    keras.callbacks.ModelCheckpoint("models/baseline_basic_model.keras", save_best_only=True)
]

Evalute it on the train dataset (commented out as the model that resulted from this will be used)

In [10]:
import os

is_baseline_model_created = os.path.exists("models/baseline_basic_model.keras")

if is_baseline_model_created != True:
    history_baseline_basic_model = baseline_basic_model.fit(
        data_inputs,
        data_outputs,
        epochs=10,
        validation_data=(data_inputs_test, data_outputs_test),
        callbacks=baseline_basic_model_callbacks
    )

    print(f"Train MAE: {min(history_baseline_basic_model.history['mae'])}")
    print(f"Test MAE: {min(history_baseline_basic_model.history['val_mae'])}")


Epoch 1/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 18840.6426 - mae: 134.6432 - val_loss: 24586.9043 - val_mae: 156.7519
Epoch 2/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 770us/step - loss: 10301.8418 - mae: 99.4485 - val_loss: 12629.3311 - val_mae: 112.2938
Epoch 3/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step - loss: 4661.0854 - mae: 66.3411 - val_loss: 4289.7793 - val_mae: 65.2898
Epoch 4/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 686us/step - loss: 1019.1379 - mae: 28.4570 - val_loss: 124.6501 - val_mae: 9.5425
Epoch 5/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 686us/step - loss: 44.5046 - mae: 4.8606 - val_loss: 44.1235 - val_mae: 5.5534
Epoch 6/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 695us/step - loss: 40.6146 - mae: 4.6134 - val_loss: 44.2196 - val_mae: 5.4827
Epoch 7/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 764us/step - loss: 40.5331 - mae: 4.6076 - val_loss: 45.3150 - val_mae: 5.5058
Epoch 8/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 667us/step - loss: 40.5508 - mae: 4.6135 - val_loss: 44.4357 - val_mae: 5.48

In [11]:
baseline_basic_model_loaded = keras.models.load_model("models/baseline_basic_model.keras")
print(f"Test MAE: {baseline_basic_model.evaluate(data_inputs_test, data_outputs_test)[1]}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.4467 - mae: 5.4843
Test MAE: 5.484278678894043


## Convolutional model

The Convolutional model will be based on the code provided in Chapter 10.2.4 (Deep Learning For Timeseries, Let’s try a 1D convolutional model) of the book Deep Learning with Python by Francois Chollet [3].

In [12]:
def build_convolutional_model():
    model = models.Sequential()
    model.add(layers.Conv1D(8, 24, input_shape=(sequence_length, 1), activation=activations.relu))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(8, 12, activation=activations.relu))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(8, 6, activation=activations.relu))
    model.add(layers.GlobalAveragePooling1D())
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

convolutional_model = build_convolutional_model()

/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 127, 8)         │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 52, 8)          │           776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 26, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 21, 8)          │           392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,377 (5.38 KB)

 Trainable params: 1,377 (5.38 KB)

 Non-trainable params: 0 (0.00 B)

Keras checkpoint to save model with best performance

In [13]:
convolutional_model_callbacks = [
    keras.callbacks.ModelCheckpoint("models/convolutional_model.keras", save_best_only=True)
]

Evalute it on the train dataset (commented out as the model that resulted from this will be used)

In [14]:
is_convolutional_model_created = os.path.exists("models/convolutional_model.keras")

if is_convolutional_model_created != True:
    history_convolutional_model = convolutional_model.fit(
        data_inputs,
        data_outputs,
        epochs=10,
        validation_data=(data_inputs_test, data_outputs_test),
        callbacks=convolutional_model_callbacks
    )
    convolutional_model_loaded = keras.models.load_model("models/convolutional_model.keras")
    print(f"Test MAE: {convolutional_model.evaluate(data_inputs_test, data_outputs_test)[1]}")

Epoch 1/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 2315.7620 - mae: 27.5588 - val_loss: 201.9394 - val_mae: 12.3732
Epoch 2/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.5813 - mae: 5.1599 - val_loss: 45.6889 - val_mae: 5.8932
Epoch 3/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.5420 - mae: 5.1302 - val_loss: 67.3208 - val_mae: 6.8222
Epoch 4/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.7206 - mae: 5.1395 - val_loss: 69.3133 - val_mae: 6.8648
Epoch 5/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.4896 - mae: 5.1428 - val_loss: 63.5471 - val_mae: 6.5724
Epoch 6/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 43.3499 - mae: 5.0612 - val_loss: 100.7305 - val_mae: 8.2129
Epoch 7/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.9797 - mae: 5.1430 - val_loss: 64.6414 - val_mae: 6.7053
Epoch 8/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.3617 - mae: 5.0951 - val_loss: 187.7713 - val_mae: 11.8585
Epoch 9/10
146/146 ━━━━━━━━━━━━━

As expected, the model performed worse than the baseline one (Dense model).

## KerasTuner

The model structure is the results of many experiments previously performed, but in order to not have hyperparameters, such as units and the activation functions, be aribtrary, the KerasTuner is used to obtain the best model possible.

The code is based on the code provided in the Keras documentation for the KerasTuner [4].

In [15]:
from keras import models
from keras import layers

def build_lstm_model(hp):
    model = models.Sequential()
    model.add(
        layers.LSTM(
            units = hp.Int("units", min_value=16, max_value=64, step=4),
            input_shape=(sequence_length, 1)
        )
    )
    
    for i in range(2):
        model.add(
            layers.Dense(
                units = hp.Int(f"units_{i}", min_value=4, max_value=56, step=4),
                activation = hp.Choice("activation", ["relu", "linear"])
            )
        )
    
    model.add(
        layers.Dense(
            units = 1
        )
    )
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

In [18]:
import keras_tuner

tuner = keras_tuner.RandomSearch(
    hypermodel=build_lstm_model,
    objective="val_mae",
    max_trials=10,
    executions_per_trial=5,
    overwrite=True,
    directory="tuner_results/final_project_tuner_ml_process_results",
    project_name="final_project_tuner_ml_process"
)

tuner.search_space_summary()

Search space summary
Default search space size: 4
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 4, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 4, 'max_value': 56, 'step': 4, 'sampling': 'linear'}
activation (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'linear'], 'ordered': False}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 4, 'max_value': 56, 'step': 4, 'sampling': 'linear'}


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# check if ml_process_tuner_best_model exists
is_ml_process_tuner_best_model_created = os.path.exists("models/ml_process_tuner_best_model.keras")

# if it does not exists run the tuner
if is_ml_process_tuner_best_model_created != True:
    tuner.search(data_inputs, data_outputs, epochs=20, validation_data=(data_inputs_test, data_outputs_test))
    best_model = tuner.get_best_models(num_models=1)[0]
    best_model.summary()

    # save model
    best_model.save("models/ml_process_tuner_best_model.keras")

Trial 10 Complete [00h 04m 46s]
val_mae: 1.8065624713897706

Best val_mae So Far: 1.716127371788025
Total elapsed time: 00h 35m 34s


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 11 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 28)             │           924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            29 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,929 (77.85 KB)

 Trainable params: 19,929 (77.85 KB)

 Non-trainable params: 0 (0.00 B)

## About the code

The timeseries code is based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

The baseline model code is based on the code provided in Chapter 10.2.3 (Deep Learning For Timeseries, Let’s try a basic machine learning model) of the book Deep Learning with Python by Francois Chollet [2].

The Convolutional model code is based on the code provided in Chapter 10.2.4 (Deep Learning For Timeseries, Let’s try a 1D convolutional model) of the book Deep Learning with Python by Francois Chollet [3].

The KerasTuner code is based on the code provided in the Keras documentation for the KerasTuner [4].

## References

1- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Preparing the data. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_5

2- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Let’s try a basic machine learning model. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_7

3- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Let’s try a 1D convolutional model. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_8

4- Keras. Getting started with KerasTuner. Retrieved from https://keras.io/keras_tuner/getting_started/